# 7 — OpenTargets edges and evidence

Purpose: reproduce OpenTargets-derived disease/gene/molecule/pathway/interaction relations while preserving source dataset and row-level evidence provenance.

Default mode is sample/read-only. Heavy downloads and writes require explicit flags. OpenTargets gene/target rows stay gene-level; protein relations require direct protein/isoform endpoints. ENCODE-rE2G/enhancer predictions must keep biosample/context and feature provenance.

In [ ]:
from __future__ import annotations
import json, os, sys
from dataclasses import asdict, is_dataclass
from pathlib import Path
import duckdb
import pandas as pd
import pyarrow.parquet as pq
try:
    from IPython import get_ipython
    if get_ipython() is None:
        raise RuntimeError("not running inside IPython")
    from IPython.display import display
except Exception:  # pragma: no cover - plain Python fallback for lightweight validation
    def display(obj):
        print(obj)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "manage_db").exists():
    REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SAMPLE_MODE = os.environ.get("JOUVENCE_NOTEBOOK_SAMPLE_MODE", os.environ.get("TXGNN_NOTEBOOK_SAMPLE_MODE", "1")) != "0"
RUN_BUILD = os.environ.get("JOUVENCE_NOTEBOOK_RUN_BUILD", os.environ.get("TXGNN_NOTEBOOK_RUN_BUILD", "0")) == "1"
RUN_FULL_VALIDATION = os.environ.get("JOUVENCE_NOTEBOOK_FULL_VALIDATION", os.environ.get("TXGNN_NOTEBOOK_FULL_VALIDATION", "0")) == "1"
VERIFIED_KG_ROOT = Path(os.environ.get("JOUVENCE_VERIFIED_KG_ROOT", os.environ.get("TXGNN_VERIFIED_KG_ROOT", "/Users/jkobject/mnt/gcs/jouvencekb-kg/v2")))
LOCAL_SAMPLE_KG_ROOT = Path(os.environ.get("JOUVENCE_LOCAL_SAMPLE_KG_ROOT", os.environ.get("TXGNN_LOCAL_SAMPLE_KG_ROOT", REPO_ROOT / "artifacts" / "cache" / "t_fc4ede1f" / "kg-v2")))
DEFAULT_KG_ROOT = Path(os.environ.get("JOUVENCE_KG_ROOT", os.environ.get("TXGNN_KG_ROOT", str(VERIFIED_KG_ROOT if VERIFIED_KG_ROOT.exists() else LOCAL_SAMPLE_KG_ROOT))))
DATA_DIR = Path(os.environ.get("JOUVENCE_DATA_DIR", os.environ.get("TXGNN_DATA_DIR", str(REPO_ROOT / "data"))))
OT_DIR = DATA_DIR / "opentargets"
BUILD_KG_ROOT = DATA_DIR / "kg"
print(f"repo={REPO_ROOT}")
print(f"sample_mode={SAMPLE_MODE} run_build={RUN_BUILD} run_full_validation={RUN_FULL_VALIDATION}")
print(f"default_kg_root={DEFAULT_KG_ROOT}")
print(f"verified_kg_exists={VERIFIED_KG_ROOT.exists()} local_sample_kg_root={LOCAL_SAMPLE_KG_ROOT} data_dir={DATA_DIR}")

In [ ]:
from manage_db import kg_storage
from manage_db.audit_edge_evidence import audit_edge_evidence
from manage_db.ingest_opentargets import (
    run as run_opentargets_ingest,
    ingest_interactions, ingest_evidence, ingest_indication, ingest_mechanism_of_action,
    ingest_target_essentiality, ingest_disease_phenotype, ingest_expression,
    ingest_pharmacogenomics, ingest_variant_protein_changes, ingest_evidence_backed_variants, ingest_enhancers,
)
try:
    from txdata_download import get_latest_opentargets_release, list_opentargets_datasets, download_opentargets_datasets
except Exception as exc:
    get_latest_opentargets_release = list_opentargets_datasets = download_opentargets_datasets = None
    print(f"txdata_download helpers unavailable: {exc}")
OPENTARGETS_DATASET_TO_FUNCTION = {
    "interaction": ingest_interactions, "evidence": ingest_evidence, "drug_indication": ingest_indication,
    "drug_mechanism_of_action": ingest_mechanism_of_action, "target_essentiality": ingest_target_essentiality,
    "disease_phenotype": ingest_disease_phenotype, "expression": ingest_expression,
    "pharmacogenomics": ingest_pharmacogenomics, "variant": ingest_variant_protein_changes,
    "known_variant": ingest_evidence_backed_variants, "enhancer_to_gene": ingest_enhancers,
}
for dataset, fn in OPENTARGETS_DATASET_TO_FUNCTION.items():
    assert callable(fn), dataset

In [ ]:
def parquet_path(kind: str, name: str, root: Path = DEFAULT_KG_ROOT) -> Path:
    return root / kind / f"{name}.parquet"

def parquet_exists(kind: str, name: str, root: Path = DEFAULT_KG_ROOT) -> bool:
    return parquet_path(kind, name, root).exists()

def parquet_rows(path: Path) -> int | None:
    if not path.exists():
        return None
    return pq.ParquetFile(path).metadata.num_rows

def relation_inventory(relations: list[str], root: Path = DEFAULT_KG_ROOT) -> pd.DataFrame:
    rows = []
    for relation in relations:
        edge = parquet_path("edges", relation, root)
        evidence = parquet_path("evidence", relation, root)
        rows.append({
            "relation": relation,
            "edge_file": edge.exists(),
            "edge_rows": parquet_rows(edge),
            "evidence_file": evidence.exists(),
            "evidence_rows": parquet_rows(evidence),
        })
    return pd.DataFrame(rows)

def duckdb_relation_summary(relation: str, root: Path = DEFAULT_KG_ROOT) -> pd.DataFrame:
    evidence_path = parquet_path("evidence", relation, root)
    if not evidence_path.exists():
        return pd.DataFrame([{"relation": relation, "status": "missing evidence parquet", "path": str(evidence_path)}])
    con = duckdb.connect(":memory:")
    try:
        cols = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{evidence_path.as_posix()}')").df()["column_name"].tolist()
        dimensions = [c for c in ["source", "source_dataset", "evidence_type", "predicate", "direction"] if c in cols]
        if not dimensions:
            return pd.DataFrame([{"relation": relation, "status": "no standard provenance columns", "path": str(evidence_path)}])
        select_dims = ", ".join(dimensions)
        query = f"""
            SELECT {select_dims}, count(*) AS rows
            FROM read_parquet('{evidence_path.as_posix()}')
            GROUP BY {select_dims}
            ORDER BY rows DESC
            LIMIT 50
        """
        return con.execute(query).df()
    finally:
        con.close()

def edge_evidence_key_anti_join(relation: str, root: Path = DEFAULT_KG_ROOT) -> pd.DataFrame:
    edge_path = parquet_path("edges", relation, root)
    evidence_path = parquet_path("evidence", relation, root)
    if not edge_path.exists() or not evidence_path.exists():
        return pd.DataFrame([{"relation": relation, "status": "missing edge or evidence parquet", "edge_path": str(edge_path), "evidence_path": str(evidence_path)}])
    con = duckdb.connect(":memory:")
    try:
        query = f"""
        WITH edge_keys AS (
            SELECT relation || '|' || x_id || '|' || y_id AS edge_key
            FROM read_parquet('{edge_path.as_posix()}')
        ), evidence_keys AS (
            SELECT COALESCE(edge_key, relation || '|' || x_id || '|' || y_id) AS edge_key
            FROM read_parquet('{evidence_path.as_posix()}')
        )
        SELECT
          (SELECT count(*) FROM edge_keys) AS edge_rows,
          (SELECT count(*) FROM evidence_keys) AS evidence_rows,
          (SELECT count(*) FROM edge_keys e WHERE NOT EXISTS (SELECT 1 FROM evidence_keys v WHERE v.edge_key = e.edge_key)) AS edges_without_evidence,
          (SELECT count(*) FROM evidence_keys v WHERE NOT EXISTS (SELECT 1 FROM edge_keys e WHERE e.edge_key = v.edge_key)) AS evidence_without_edge
        """
        return con.execute(query).df()
    finally:
        con.close()

def endpoint_type_summary(relation: str, root: Path = DEFAULT_KG_ROOT) -> pd.DataFrame:
    edge_path = parquet_path("edges", relation, root)
    if not edge_path.exists():
        return pd.DataFrame([{"relation": relation, "status": "missing edge parquet", "path": str(edge_path)}])
    con = duckdb.connect(":memory:")
    try:
        return con.execute(f"""
            SELECT x_type, y_type, count(*) AS rows
            FROM read_parquet('{edge_path.as_posix()}')
            GROUP BY 1, 2
            ORDER BY rows DESC
        """).df()
    finally:
        con.close()

## Release/source inventory and guarded build cells

In [ ]:
if get_latest_opentargets_release is not None and list_opentargets_datasets is not None:
    try:
        release = os.environ.get("OPENTARGETS_RELEASE") or get_latest_opentargets_release()
        datasets = list_opentargets_datasets(release)
        print(f"selected OpenTargets release: {release}")
        display(pd.DataFrame({"dataset": datasets}).head(50))
    except Exception as exc:
        print(f"OpenTargets release/dataset discovery failed; use pinned local sources instead: {exc}")

SELECTED_DATASETS = list(OPENTARGETS_DATASET_TO_FUNCTION)
if RUN_BUILD:
    run_opentargets_ingest(DATA_DIR, datasets=SELECTED_DATASETS, release=os.environ.get("OPENTARGETS_RELEASE", "latest"), download=os.environ.get("JOUVENCE_NOTEBOOK_DOWNLOAD", os.environ.get("TXGNN_NOTEBOOK_DOWNLOAD", "0")) == "1", workers=int(os.environ.get("JOUVENCE_DOWNLOAD_WORKERS", os.environ.get("TXGNN_DOWNLOAD_WORKERS", "4"))))
else:
    display(pd.DataFrame([{"dataset": ds, "builder": OPENTARGETS_DATASET_TO_FUNCTION[ds].__name__, "status": "skipped; set JOUVENCE_NOTEBOOK_RUN_BUILD=1"} for ds in SELECTED_DATASETS]))

## Dataset → relation/evidence contract

In [ ]:
OT_RELATION_MAP = pd.DataFrame([
    {"dataset":"interaction", "native_assertion":"target/gene or gene-product interaction", "canonical_relations":"gene_interacts_gene; protein_interacts_protein only for protein-native endpoints", "evidence_detail":"source subdatabase, interaction type, score, direction/sign, PMID/source IDs"},
    {"dataset":"evidence", "native_assertion":"target/gene associated with disease/trait", "canonical_relations":"disease_associated_gene, mutation_associated_disease, mutation_associated_gene", "evidence_detail":"datatype, score, study/locus IDs, p-value/effect where present"},
    {"dataset":"drug_indication", "native_assertion":"molecule has clinical indication for disease", "canonical_relations":"molecule_treats_disease", "evidence_detail":"max phase, status, disease, source record"},
    {"dataset":"drug_mechanism_of_action", "native_assertion":"molecule targets OpenTargets target/gene", "canonical_relations":"molecule_targets_gene; molecule_targets_protein only for direct protein endpoints", "evidence_detail":"action_type, mechanism, target class, source record"},
    {"dataset":"target_essentiality", "native_assertion":"gene essentiality/dependency/expression in cell line", "canonical_relations":"cell_line_gene_essentiality, cell_line_expresses_gene", "evidence_detail":"screen/study, score/effect, cell line context"},
    {"dataset":"disease_phenotype", "native_assertion":"disease associated with phenotype", "canonical_relations":"disease_has_phenotype", "evidence_detail":"HPO/EFO IDs, frequency/severity/source fields"},
    {"dataset":"expression", "native_assertion":"RNA expression in tissue/cell type", "canonical_relations":"tissue_expresses_gene, cell_type_expresses_gene", "evidence_detail":"TPM/value, level, tissue/cell context; no protein projection"},
    {"dataset":"pharmacogenomics", "native_assertion":"mutation/variant affects molecule response", "canonical_relations":"mutation_affects_molecule_response", "evidence_detail":"drug response predicate, source record, score/effect"},
    {"dataset":"variant/known_variant", "native_assertion":"variant changes protein or links to disease/gene evidence", "canonical_relations":"mutation_changes_protein, mutation_associated_disease, mutation_associated_gene", "evidence_detail":"variant ID, consequence, score, study/locus IDs"},
    {"dataset":"enhancer_to_gene", "native_assertion":"context-specific enhancer→gene prediction", "canonical_relations":"enhancer_regulates_gene", "evidence_detail":"biosample, rE2G score, DNase, Hi-C/contact, distance, QC/features"},
])
display(OT_RELATION_MAP)

## Cached relation inventory, provenance summaries, and support audits

In [ ]:
OT_RELATIONS = ["gene_interacts_gene", "molecule_targets_gene", "molecule_treats_disease", "disease_associated_gene", "mutation_associated_disease", "mutation_associated_gene", "mutation_changes_protein", "mutation_affects_molecule_response", "enhancer_regulates_gene", "tissue_expresses_gene", "cell_type_expresses_gene", "cell_line_gene_essentiality"]
display(relation_inventory(OT_RELATIONS))
for relation in ["gene_interacts_gene", "molecule_targets_gene", "mutation_associated_disease", "enhancer_regulates_gene"]:
    print(f"\n=== {relation} evidence provenance ===")
    display(duckdb_relation_summary(relation))
candidate_relations = [r for r in OT_RELATIONS if parquet_exists("edges", r) or parquet_exists("evidence", r)]
if candidate_relations:
    try:
        audit = audit_edge_evidence(DEFAULT_KG_ROOT, relations=candidate_relations)
        display(pd.DataFrame([asdict(report) | {"ok": report.ok} for report in audit.relation_reports.values()]))
    except Exception as exc:
        print(f"Repository evidence audit could not run here: {exc}")
    for relation in candidate_relations[:5]:
        display(edge_evidence_key_anti_join(relation))
        display(endpoint_type_summary(relation))
else:
    print("No cached OT relation/evidence files found. Copy targeted files from gs://jouvencekb/kg/v2 into artifacts/cache/t_fc4ede1f/kg-v2 or set JOUVENCE_KG_ROOT (legacy fallback: TXGNN_KG_ROOT) for sample mode.")
print("Protein relations require direct protein/isoform endpoints; do not project gene/RNA rows.")
print("Enhancer predictions require biosample/context and feature provenance.")